# Demo: Using Pretrained EpiAgent for Zero-Shot Cell Embeddings on Kanemaru2023 Dataset

This notebook demonstrates the use of the pretrained EpiAgent model, which has been trained on the large-scale Human-scATAC-Corpus with approximately 5 million cells and 35 billion tokens. It illustrates a zero-shot inference workflow to extract cell embeddings from the downsampled Kanemaru2023 dataset (Kanemaru2023_downsampled_10000_cells.h5ad).

Required files:

	•	pretrained_EpiAgent.pth: The pretrained EpiAgent model.
	•	Kanemaru2023_downsampled_10000_cells.h5ad: The downsampled Kanemaru2023 dataset.

These files are available at the following link: https://drive.google.com/drive/folders/1WlNykSCNtZGsUp2oG0dw3cDdVKYDR-iX?usp=sharing.

# Step 1: Data Processing (TFIDF and Tokenization)

To prepare the Kanemaru2023 dataset for EpiAgent, we perform the following preprocessing steps:

1.TFIDF Transformation: Convert discrete count data into continuous importance scores for accessible cCREs.
    
2.Tokenization: Generate cell_sentences to represent each cell as a sequence of accessible cCRE indices.

In [1]:
import random
import numpy as np
import torch
import wandb
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
wandb.init(project="zero-shot-feature-extraction", name=f"zero_shot_embeddings_Kanemaru2023_{timestamp}")

wandb: Currently logged in as: jack-naimer (jack-naimer-epfl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
x=1

In [3]:
x

1

In [4]:
import scanpy as sc
import numpy as np
from epiagent.tokenization import tokenization
from epiagent.preprocessing import global_TFIDF

# Load the dataset
# input_path = '../data/sample/raw_h5ad/Kanemaru2023_downsampled_10000_cells.h5ad'
input_path = '/scratch/naimer/Kanemaru2023/Kanemaru2023-cardiac_tissue/Kanemaru2023-cardiac_tissue-cell_by_cCRE.h5ad'
adata = sc.read_h5ad(input_path)

# Load the cCRE document frequency data
cCRE_document_frequency = np.load('/home/naimer/github/project-2-team-1-gal/EpiAgent/data/cCRE_document_frequency.npy')

# No permutations

In [ ]:
# Apply TFIDF transformation
adata_tfidf = global_TFIDF(adata, cCRE_document_frequency)

# Perform tokenization
tokenization(adata_tfidf)

# Step 2: Create Dataset and DataLoader

We create a PyTorch-compatible Dataset and DataLoader to handle tokenized cell_sentences from the processed AnnData object. Each cell is represented as a sequence of tokens with special tokens [CLS] and [SEP].

In [ ]:
from epiagent.dataset import CellDataset, collate_fn
from torch.utils.data import DataLoader

# Create the dataset
cell_sentences = adata_tfidf.obs['cell_sentences'].tolist()
cell_dataset = CellDataset(cell_sentences=cell_sentences)

# Create the DataLoader
batch_size = 15
dataloader = DataLoader(cell_dataset, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_fn)

# Step 3: Load Pretrained EpiAgent Model

The pretrained EpiAgent model (pretrained_EpiAgent.pth) is loaded for zero-shot inference to compute cell embeddings.

In [ ]:
import os
# Force tqdm to use standard progress bar instead of notebook version
os.environ['TQDM_NOTEBOOK'] = '0'

from epiagent.model import EpiAgent
import torch

# Load the pretrained model
model_path = '/home/naimer/github/project-2-team-1-gal/EpiAgent/model/pretrained_EpiAgent.pth'
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

pretrained_model = EpiAgent(vocab_size=1355449, num_layers=18, embedding_dim=512, num_attention_heads=8, max_rank_embeddings=8192, use_flash_attn=True, pos_weight_for_RLM=torch.tensor(1.), pos_weight_for_CCA=torch.tensor(1.))
pretrained_model.load_state_dict(torch.load(model_path, map_location=device))

<All keys matched successfully>

wandb: 
wandb: 🚀 View run zero_shot_embeddings_20251124_204501 at: 
wandb: Find logs at: wandb/run-20251124_204501-wa99uxrr/logs


# Step 4: Extract Cell Embeddings


The infer_cell_embeddings function is used to compute cell embeddings using the pretrained EpiAgent model.

In [ ]:
from epiagent.inference import infer_cell_embeddings

# Extract cell embeddings
cell_embeddings = infer_cell_embeddings(pretrained_model, device, dataloader)

# Step 5: UMAP Visualization

Perform UMAP visualization to project the embeddings into a 2D space.

In [ ]:
import io
from PIL import Image

def prepare_img(fig):
    """
    Save matplotlib figure to PIL Image for wandb logging with high resolution.
    
    Args:
        fig: matplotlib figure object
        
    Returns:
        PIL Image object ready for wandb.Image()
    """
    # Save figure to buffer with high DPI and no cropping for wandb
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=300, bbox_inches='tight', pad_inches=0.1, facecolor='white')
    buf.seek(0)
    img = Image.open(buf)
    return img


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Assign embeddings to the AnnData object
adata_tfidf.obsm['cell_embeddings_zero_shot'] = cell_embeddings

# UMAP visualization
sc.pp.neighbors(adata_tfidf, use_rep='cell_embeddings_zero_shot')
sc.tl.umap(adata_tfidf)

# Plot UMAP with original cell types and capture the figure
fig = sc.pl.umap(adata_tfidf, color='cell_type', return_fig=True, show=True, title='Cell embeddings (true labels)')

# Customize legend to show "cell type" as description
if fig is not None:
    axes = fig.axes if hasattr(fig, 'axes') else [ax for ax in fig.get_axes()]
    for ax in axes:
        legend = ax.get_legend()
        if legend is not None:
            legend.set_title('Cell type')
plt.show()

# Convert figure to PIL Image for wandb using reusable function
img = prepare_img(fig)

wandb.log({"umap_cell_types_no_shuffling_true_labels": wandb.Image(img)})
plt.close(fig)


In [ ]:
pretrained_model.criterion_CCA.pos_weight

In [ ]:
print(f"Number of cell types: {len(adata_tfidf.obs['cell_type'].unique())}")

In [ ]:
# Perform leiden clustering
# Note: neighbors are already computed in cell 16 using the embeddings
# We'll use a default resolution of 0.5, which can be adjusted based on desired granularity
sc.tl.leiden(adata_tfidf, resolution=0.6, key_added='leiden', random_state=42)

# Print the number of clusters found
n_clusters = len(adata_tfidf.obs['leiden'].unique())
print(f"Number of Leiden clusters: {n_clusters}")

# Visualize the Leiden clusters on UMAP
fig = sc.pl.umap(adata_tfidf, color='leiden', legend_loc='on data', title='Leiden Clustering (no shuffling)', return_fig=True, show=True)
plt.show()
img = prepare_img(fig)
wandb.log({"umap_cell_types_no_shuffling_leiden_clustering": wandb.Image(img)})
plt.close(fig)

In [ ]:
# Calculate NMI and ARI scores
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Get the true labels (cell types) and predicted labels (Leiden clusters)
true_labels = adata_tfidf.obs['cell_type'].values
predicted_labels = adata_tfidf.obs['leiden'].values

# Calculate ARI (Adjusted Rand Index)
ari_score = adjusted_rand_score(true_labels, predicted_labels)

# Calculate NMI (Normalized Mutual Information)
nmi_score = normalized_mutual_info_score(true_labels, predicted_labels)

# Print the results
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi_score:.4f}")
print(f"\nNumber of true cell types: {len(adata_tfidf.obs['cell_type'].unique())}")
print(f"Number of predicted clusters: {len(adata_tfidf.obs['leiden'].unique())}")

wandb.log({
    "ARI_no_shuffling": ari_score,
    "NMI_no_shuffling": nmi_score
})

# Permute labels

In [ ]:
import scanpy as sc
import numpy as np
from epiagent.tokenization import tokenization
from epiagent.preprocessing import global_TFIDF_with_shuffling

# Apply TFIDF transformation
adata_tfidf_perm = global_TFIDF_with_shuffling(adata, cCRE_document_frequency)

# Perform tokenization
tokenization(adata_tfidf_perm)

In [ ]:
from epiagent.dataset import CellDataset, collate_fn
from torch.utils.data import DataLoader

# Create the dataset
cell_sentences_perm = adata_tfidf_perm.obs['cell_sentences'].tolist()
cell_dataset_perm = CellDataset(cell_sentences=cell_sentences_perm)

# Create the DataLoader
batch_size = 15
dataloader_perm = DataLoader(cell_dataset_perm, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_fn)

In [ ]:
from epiagent.inference import infer_cell_embeddings

# Extract cell embeddings
cell_embeddings_perm = infer_cell_embeddings(pretrained_model, device, dataloader_perm)

In [ ]:
import scanpy as sc

# Assign embeddings to the AnnData object
adata_tfidf_perm.obsm['cell_embeddings_zero_shot'] = cell_embeddings_perm

# UMAP visualization
sc.pp.neighbors(adata_tfidf_perm, use_rep='cell_embeddings_zero_shot')
sc.tl.umap(adata_tfidf_perm)

# Plot UMAP with original cell types
fig = sc.pl.umap(adata_tfidf_perm, color='cell_type', return_fig=True, show=True)

if fig is not None:
    axes = fig.axes if hasattr(fig, 'axes') else [ax for ax in fig.get_axes()]
    for ax in axes:
        legend = ax.get_legend()
        if legend is not None:
            legend.set_title('Cell type')

plt.show()
img = prepare_img(fig)
wandb.log({"umap_cell_types_permuted_labels": wandb.Image(img)})
plt.close(fig)

In [ ]:
# Assign embeddings to adata_perm and compute neighbors for Leiden clustering
adata_tfidf_perm.obsm['cell_embeddings_zero_shot'] = cell_embeddings_perm

# Compute neighbors graph (required for Leiden clustering)
# sc.pp.neighbors(adata_tfidf_perm, use_rep='cell_embeddings_zero_shot') # TODO: can probably remove this

# Compute UMAP for visualization
# sc.tl.umap(adata_tfidf_perm) # TODO: can probably remove this

# Perform Leiden clustering
sc.tl.leiden(adata_tfidf_perm, resolution=0.5, key_added='leiden', random_state=42)

# Print the number of clusters found
n_clusters = len(adata_tfidf_perm.obs['leiden'].unique())
print(f"Number of Leiden clusters: {n_clusters}")

# Visualize the Leiden clusters on UMAP
fig = sc.pl.umap(adata_tfidf_perm, color='leiden', legend_loc='on data', title='Leiden Clustering', return_fig=True, show=True)
plt.show()
img = prepare_img(fig)
wandb.log({"umap_cell_types_permuted_labels_leiden_clustering": wandb.Image(img)})
plt.close(fig)

In [ ]:
# Calculate NMI and ARI scores
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Get the true labels (cell types) and predicted labels (Leiden clusters)
true_labels = adata_tfidf_perm.obs['cell_type'].values
predicted_labels = adata_tfidf_perm.obs['leiden'].values

# Calculate ARI (Adjusted Rand Index)
ari_score = adjusted_rand_score(true_labels, predicted_labels)

# Calculate NMI (Normalized Mutual Information)
nmi_score = normalized_mutual_info_score(true_labels, predicted_labels)

# Print the results
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi_score:.4f}")
print(f"\nNumber of true cell types: {len(adata_tfidf_perm.obs['cell_type'].unique())}")
print(f"Number of predicted clusters: {len(adata_tfidf_perm.obs['leiden'].unique())}")

wandb.log({
    "ARI_permuted_labels": ari_score,
    "NMI_permuted_labels": nmi_score
})


# Completely permuted labels

In [ ]:
import scanpy as sc
import numpy as np
from epiagent.tokenization import tokenization
from epiagent.preprocessing import global_TFIDF_with_complete_shuffling

# Load the dataset
# input_path = '../data/sample/raw_h5ad/Kanemaru2023_downsampled_10000_cells.h5ad'
# Note: copy() creates a copy of the AnnData structure. The global_TFIDF_with_shuffling 
# function also creates its own internal copy, so this ensures adata_perm is separate from adata.
# adata_perm = adata.copy()

# Load the cCRE document frequency data
# cCRE_document_frequency = np.load('../data/cCRE_document_frequency.npy')

# Apply TFIDF transformation
adata_tfidf_perm_complete = global_TFIDF_with_complete_shuffling(adata, cCRE_document_frequency)

# Perform tokenization
tokenization(adata_tfidf_perm_complete)

In [ ]:
from epiagent.dataset import CellDataset, collate_fn
from torch.utils.data import DataLoader

# Create the dataset
cell_sentences_perm_complete = adata_tfidf_perm_complete.obs['cell_sentences'].tolist()
cell_dataset_perm_complete = CellDataset(cell_sentences=cell_sentences_perm_complete)

# Create the DataLoader
batch_size = 15
dataloader_perm_complete = DataLoader(cell_dataset_perm_complete, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_fn)

In [ ]:
from epiagent.inference import infer_cell_embeddings

# Extract cell embeddings
cell_embeddings_perm_complete = infer_cell_embeddings(pretrained_model, device, dataloader_perm_complete)

In [ ]:
import scanpy as sc

# Assign embeddings to the AnnData object
adata_tfidf_perm_complete.obsm['cell_embeddings_zero_shot'] = cell_embeddings_perm_complete

# UMAP visualization
sc.pp.neighbors(adata_tfidf_perm_complete, use_rep='cell_embeddings_zero_shot')
sc.tl.umap(adata_tfidf_perm_complete)

# Plot UMAP with original cell types
fig = sc.pl.umap(adata_tfidf_perm_complete, color='cell_type', return_fig=True, show=True)

if fig is not None:
    axes = fig.axes if hasattr(fig, 'axes') else [ax for ax in fig.get_axes()]
    for ax in axes:
        legend = ax.get_legend()
        if legend is not None:
            legend.set_title('Cell type')

plt.show()
img = prepare_img(fig)
wandb.log({"umap_cell_types_complete_shuffling": wandb.Image(img)})
plt.close(fig)

# # Save the processed AnnData
# output_path = '../data/sample/processed_h5ad/Kanemaru2023_downsampled_10000_cells_EpiAgent_zero_shot_outputs.h5ad'
# adata_tfidf.write(output_path)
# print(f"Processed AnnData saved at {output_path}")

In [ ]:
# Assign embeddings to adata_perm and compute neighbors for Leiden clustering
adata_tfidf_perm_complete.obsm['cell_embeddings_zero_shot'] = cell_embeddings_perm_complete

# Compute neighbors graph (required for Leiden clustering)
# sc.pp.neighbors(adata_tfidf_perm_complete, use_rep='cell_embeddings_zero_shot')

# Compute UMAP for visualization
# sc.tl.umap(adata_tfidf_perm_complete)

# Perform Leiden clustering
sc.tl.leiden(adata_tfidf_perm_complete, resolution=1.2, key_added='leiden', random_state=42)

# Print the number of clusters found
n_clusters = len(adata_tfidf_perm_complete.obs['leiden'].unique())
print(f"Number of Leiden clusters: {n_clusters}")

# Visualize the Leiden clusters on UMAP
fig = sc.pl.umap(adata_tfidf_perm_complete, color='leiden', legend_loc='on data', title='Leiden Clustering', return_fig=True, show=True)
plt.show()
img = prepare_img(fig)
wandb.log({"umap_cell_types_complete_shuffling_leiden_clustering": wandb.Image(img)})
plt.close(fig)

In [ ]:
# Calculate NMI and ARI scores
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Get the true labels (cell types) and predicted labels (Leiden clusters)
true_labels = adata_tfidf_perm_complete.obs['cell_type'].values
predicted_labels = adata_tfidf_perm_complete.obs['leiden'].values

# Calculate ARI (Adjusted Rand Index)
ari_score = adjusted_rand_score(true_labels, predicted_labels)

# Calculate NMI (Normalized Mutual Information)
nmi_score = normalized_mutual_info_score(true_labels, predicted_labels)

# Print the results
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi_score:.4f}")
print(f"\nNumber of true cell types: {len(adata_tfidf_perm_complete.obs['cell_type'].unique())}")
print(f"Number of predicted clusters: {len(adata_tfidf_perm_complete.obs['leiden'].unique())}")

wandb.log({
    "ARI_complete_shuffling": ari_score,
    "NMI_complete_shuffling": nmi_score
})
